<a href="https://colab.research.google.com/github/kaladharanalytics/AI-TechRadar/blob/main/novelty_core_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Grant Summary Analysis and Novelty Scoring

This notebook aims to demonstrate how to:
1.  Take a grant summary as input.
2.  Extract key terms from the summary.
3.  Use these terms to search PubMed via its E-utilities API to find relevant research.
4.  Calculate a 'novelty score' for the grant summary based on the retrieved research, providing justification for the score.

### 1. Grant Summary Placeholder

Since no specific grant summary was provided, I'm using a placeholder text. You can replace the content of the `grant_summary_text` variable below with your actual grant summary.

In [19]:
grant_name = """Strengthening Global
Health Security by
improving public health
capacity to detect, notify,
and respond to disease
outbreaks globally"""
grant_id = "CFCDC-RFA-JG-26-0056"
grant_summary_text = """
This notice of funding opportunity (NOFO) will enhance capacity to contain
outbreaks through partnerships with Ministries of Health (MOHs), external
stakeholders, and other governmental institutions. This NOFO prioritizes
durable Global Health Security (GHS) programs to keep Americans safer,
stronger, and more prosperous by improving public health globally. It applies
lessons learned from recent significant public health events, including
outbreaks of viral hemorrhagic fevers (VHF), highly pathogenic strains of
influenza, and the COVID-19 pandemic.
As global leaders in strengthening GHS systems, the U.S. government (USG)
will use updated frameworks to advance disease detection, notification, and
response strategies. This NOFO’s main outcomes are:
• Use time-bound evidence based frameworks, such as the 7-1-7
framework, to detect and respond to outbreaks more quickly for diseases
that threaten global health security and the health of Americans at home
and abroad.
• Support regional and country public health institutions to independently
lead preparedness efforts and create resilient, durable, and locally
owned health systems.
• Enhance data systems, lab capacity, and local field epidemiology
workforce to effectively combat disease threats of international
significance.
"""

print(f"Grant Name: {grant_name}")
print(f"Grant ID: {grant_id}")
print("Grant Summary:")
print(grant_summary_text)

Grant Name: Strengthening Global
Health Security by
improving public health
capacity to detect, notify,
and respond to disease
outbreaks globally
Grant ID: CFCDC-RFA-JG-26-0056
Grant Summary:

This notice of funding opportunity (NOFO) will enhance capacity to contain 
outbreaks through partnerships with Ministries of Health (MOHs), external 
stakeholders, and other governmental institutions. This NOFO prioritizes 
durable Global Health Security (GHS) programs to keep Americans safer, 
stronger, and more prosperous by improving public health globally. It applies 
lessons learned from recent significant public health events, including 
outbreaks of viral hemorrhagic fevers (VHF), highly pathogenic strains of 
influenza, and the COVID-19 pandemic. 
As global leaders in strengthening GHS systems, the U.S. government (USG) 
will use updated frameworks to advance disease detection, notification, and 
response strategies. This NOFO’s main outcomes are: 
• Use time-bound evidence based framewo

### 2. Install and Import Libraries

We'll need `requests` for making API calls to PubMed, `nltk` for natural language processing (specifically keyword extraction), and `xml.etree.ElementTree` for parsing the XML responses from PubMed.

In [20]:
%pip install requests nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [21]:
%pip install requests nltk

In [22]:
import requests
import xml.etree.ElementTree as ET
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# NLTK resources are now downloaded in a dedicated installation cell.

stop_words = set(stopwords.words('english'))

### 3. Extract Keywords from Grant Summary

To effectively search PubMed, we need to extract meaningful keywords from the grant summary. We'll tokenize the text, remove stop words and punctuation, and convert to lowercase.

In [23]:
def extract_keywords(text):
    tokens = word_tokenize(text.lower())
    keywords = [
        word for word in tokens
        if word.isalnum() and word not in stop_words and len(word) > 2
    ]
    return list(set(keywords)) # Return unique keywords

grant_keywords = extract_keywords(grant_summary_text)
print("Extracted Keywords from Grant Summary:", grant_keywords)

Extracted Keywords from Grant Summary: ['viral', 'local', 'enhance', 'stakeholders', 'international', 'detection', 'diseases', 'lab', 'response', 'significance', 'epidemiology', 'events', 'frameworks', 'threats', 'usg', 'funding', 'notification', 'owned', 'including', 'use', 'americans', 'learned', 'preparedness', 'governmental', 'capacity', 'public', 'framework', 'ministries', 'effectively', 'prosperous', 'strains', 'outbreaks', 'systems', 'significant', 'contain', 'programs', 'recent', 'support', 'locally', 'threaten', 'stronger', 'based', 'home', 'institutions', 'partnerships', 'lessons', 'notice', 'create', 'ghs', 'improving', 'globally', 'government', 'disease', 'field', 'highly', 'hemorrhagic', 'evidence', 'workforce', 'detect', 'opportunity', 'vhf', 'global', 'fevers', 'efforts', 'regional', 'outcomes', 'keep', 'safer', 'durable', 'data', 'nofo', 'external', 'independently', 'main', 'respond', 'resilient', 'quickly', 'health', 'combat', 'applies', 'abroad', 'security', 'lead', '

### 4. Fetch Research Information from PubMed

We will use the Entrez Programming Utilities (E-utilities) from NCBI, specifically `esearch` to find relevant article IDs and `efetch` to retrieve details like titles and abstracts. We'll search for the top 5 most relevant articles based on our extracted keywords.

In [24]:
def search_pubmed(query, retmax=5):
    # ESearch to get PMIDs
    esearch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    params = {
        'db': 'pubmed',
        'term': query,
        'retmax': retmax,
        'retmode': 'json'
    }
    response = requests.get(esearch_url, params=params)
    data = response.json()
    pmids = data['esearchresult']['idlist']

    if not pmids:
        return []

    # EFetch to get article details
    efetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    params = {
        'db': 'pubmed',
        'id': ','.join(pmids),
        'retmode': 'xml'
    }
    response = requests.get(efetch_url, params=params)
    root = ET.fromstring(response.content)

    articles = []
    for article_elem in root.findall('.//PubmedArticle'):
        title = article_elem.find('.//ArticleTitle')
        abstract_elem = article_elem.find('.//AbstractText')

        abstract = ''
        if abstract_elem is not None:
            # Handle multi-part abstracts if they exist
            abstract_parts = article_elem.findall('.//AbstractText')
            abstract = ' '.join([part.text for part in abstract_parts if part.text])

        articles.append({
            'pmid': article_elem.find('.//PMID').text if article_elem.find('.//PMID') is not None else 'N/A',
            'title': title.text if title is not None else 'No Title',
            'abstract': abstract
        })
    return articles

# Construct PubMed query from grant keywords
# The previous query was hardcoded for CRISPR-CF. We now need to build a query
# that reflects the new grant summary's topic (Global Health Security).
# We'll use the extracted grant_keywords, prioritizing key phrases if possible.

# Re-extract keywords to ensure they reflect the *latest* grant_summary_text
# (Assuming grant_keywords is updated by re-running previous cells after grant_summary_text update)
# For a more targeted search, we can combine some keywords with 'AND' and others with 'OR'.

# For the 'Global Health Security' grant, let's try to identify core concepts.
# Example: Focus on terms like 'global health security', 'outbreaks', 'public health', 'disease response'

# Since `grant_keywords` should have been re-extracted from the new grant_summary_text, we will use them.
# To make the search more precise, we can join a subset of the most relevant keywords with 'AND',
# and others with 'OR' if they represent synonyms or related concepts.

# For simplicity and to use the already extracted keywords, let's join all of them with 'AND'
# for a very specific search, or 'OR' for a broad search, then refine if needed.
# Given the breadth of 'Global Health Security', a combination might be best.
# Let's start by joining a selection of important keywords with 'AND' for specificity.

# Important keywords from the new grant summary (assuming re-extraction)
# These should be dynamically generated based on grant_keywords after re-running ab987877

# Placeholder for a more intelligent query construction, for now using 'AND' for extracted keywords.
# A more sophisticated approach would involve NLP to identify key phrases or concepts for 'AND'ing.
# For now, let's join all extracted keywords with 'AND' and see the results, or choose a few critical ones.

# Let's try combining key phrases with 'AND' and individual important keywords with 'OR'
# This will require manual selection or a more advanced NLP step which is out of scope for this immediate fix.
# For now, let's create a query using a few central concepts joined by 'AND'.

# Re-evaluating based on the new grant summary content:
# Core concepts might include: 'global health security', 'public health capacity', 'disease outbreaks', 'response strategies'

# Let's construct a query from a selection of likely important keywords extracted from the *new* grant summary.
# This assumes `grant_keywords` will be updated when `ab987877` is re-run.

# Example of how to construct a better query for the *new* grant:
# It's better to explicitly extract key phrases if not done by `extract_keywords`.

# Temporarily, let's create a query that is broad enough using some of the expected new keywords.
# A direct `OR` of all `grant_keywords` can be too broad, `AND` too restrictive.

# Let's try to get a more sensible query for 'Global Health Security'
# based on the content. A good strategy is to use phrases for core concepts.

# This part needs to be re-thought based on what keywords are actually extracted from the *new* summary.
# For the initial fix, let's make a basic query.

# Placeholder for a refined query construction. For now, let's use some general terms from the new summary.
# The `grant_keywords` variable would ideally be regenerated from the new `grant_summary_text`.
# Once `grant_keywords` is correctly updated after re-running `ab987877`,
# this section will need another update to use those new keywords effectively.

# For now, let's construct a simple query using 'AND' with the existing, but *incorrect*, grant_keywords.
# The user will need to re-run keyword extraction (`ab987877`) first.

# Let's attempt a dynamic query construction based on the *expected* new keywords.
# Assuming grant_keywords contains relevant terms for 'Global Health Security'
# after re-execution of ab987877, we will join them with 'OR' initially, then refine.
# A more robust solution would involve keyword prioritization or phrase extraction.

# Joining current `grant_keywords` (which are still from the old summary) with 'OR' will be too broad.
# Joining with 'AND' will be too restrictive.

# The best immediate approach is to use a few strong terms from the new summary and combine them.
# User must re-run `ab987877` first to update `grant_keywords`.

# Let's make a query that is specifically tuned for the 'Global Health Security' grant.
# This is a manual update given the context change. The previous `core_concepts_query_parts` are wrong.

# New query based on the 'Global Health Security' summary:
pubmed_query_parts = [
    '("global health security" OR "health security")',
    '("public health capacity" OR "public health")',
    '("disease outbreaks" OR outbreaks OR epidemic OR pandemic)',
    '("response strategies" OR response OR preparedness OR resilience)'
]
pubmed_query = " AND ".join(pubmed_query_parts)

print(f"Refined PubMed Query: {pubmed_query}")

pubmed_articles = search_pubmed(pubmed_query, retmax=5)

print("\n--- Top 5 Relevant PubMed Articles ---")
if pubmed_articles:
    for i, article in enumerate(pubmed_articles):
        print(f"\nArticle {i+1}:")
        print(f"  PMID: {article['pmid']}")
        print(f"  Title: {article['title']}")
else:
    print("No articles found for the given query.")

Refined PubMed Query: ("global health security" OR "health security") AND ("public health capacity" OR "public health") AND ("disease outbreaks" OR outbreaks OR epidemic OR pandemic) AND ("response strategies" OR response OR preparedness OR resilience)

--- Top 5 Relevant PubMed Articles ---

Article 1:
  PMID: 42468226
  Title: Conference report: airway mucosal sampling and immune analysis.

Article 2:
  PMID: 42465635
  Title: From innovation network to public health benefit: evolutionary characteristics and optimization of the biomedical industry innovation network in the Beijing-Tianjin-Hebei region.

Article 3:
  PMID: 42461270
  Title: The Role of Artificial Intelligence and Machine Learning in Predictive Virology: Forecasting, Tracking, and Combating Viral Threats.

Article 4:
  PMID: 42460684
  Title: Integrating Oral Health Into Pandemic Preparedness and Response: Missed Opportunities and Strategic Imperatives for Global Health System Resilience and Security.

Article 5:
  PMI

In [25]:
print("\n--- Top 5 Relevant PubMed Articles (from Refined Search) ---")
if pubmed_articles:
    for i, article in enumerate(pubmed_articles):
        print(f"\nArticle {i+1}:")
        print(f"  PMID: {article['pmid']}")
        print(f"  Title: {article['title']}")
else:
    print("No articles found for the given query.")


--- Top 5 Relevant PubMed Articles (from Refined Search) ---

Article 1:
  PMID: 42468226
  Title: Conference report: airway mucosal sampling and immune analysis.

Article 2:
  PMID: 42465635
  Title: From innovation network to public health benefit: evolutionary characteristics and optimization of the biomedical industry innovation network in the Beijing-Tianjin-Hebei region.

Article 3:
  PMID: 42461270
  Title: The Role of Artificial Intelligence and Machine Learning in Predictive Virology: Forecasting, Tracking, and Combating Viral Threats.

Article 4:
  PMID: 42460684
  Title: Integrating Oral Health Into Pandemic Preparedness and Response: Missed Opportunities and Strategic Imperatives for Global Health System Resilience and Security.

Article 5:
  PMID: 42459476
  Title: Financing fragility and pandemic preparedness in Central Asia: a policy review of aid volatility, donor dynamics, and health system resilience.


### 5. Calculate Novelty Score with Justification

To calculate a 'novelty score', we can assess how unique the core concepts of the grant summary are compared to the retrieved PubMed articles. A simple approach is to calculate the proportion of keywords from the grant summary that do not appear in the titles or abstracts of the top research articles.

A higher novelty score would indicate that the grant proposes concepts or combinations of concepts that are less commonly found in the existing top research, suggesting a more novel direction.

In [26]:
def calculate_novelty_score(grant_keywords, pubmed_articles):
    if not pubmed_articles:
        return 1.0, "No existing research found; potentially very novel or query too specific."

    all_pubmed_keywords = set()
    for article in pubmed_articles:
        article_text = article['title'] + ' ' + article['abstract']
        all_pubmed_keywords.update(extract_keywords(article_text))

    # Keywords in grant summary not found in any of the top PubMed articles
    unique_to_grant = [kw for kw in grant_keywords if kw not in all_pubmed_keywords]

    if not grant_keywords:
        return 0.0, "No keywords extracted from grant summary."

    # Novelty score: proportion of grant keywords that are unique to the grant
    novelty_score = len(unique_to_grant) / len(grant_keywords)

    justification = f"The novelty score is calculated as the proportion of unique keywords from the grant summary that were not found in the titles or abstracts of the top 5 PubMed articles. "
    if novelty_score > 0.7:
        justification += "This high score suggests that the grant explores highly unique concepts or a novel combination of established ideas, indicating a strong potential for groundbreaking research."
    elif novelty_score > 0.3:
        justification += "This moderate score indicates that the grant introduces some novel elements while also building upon existing research. It strikes a balance between innovation and foundational knowledge."
    else:
        justification += "This low score suggests significant  overlap with existing research. The grant might be incremental or exploratory within a well-studied domain. Reviewing the specific unique keywords (if any) and common keywords could provide more insight."

    if unique_to_grant:
        justification += f"\nKeywords unique to the grant: {', '.join(unique_to_grant)}."
    else:
        justification += f"\nAll grant keywords were found in the top PubMed articles, indicating a strong connection to current research."

    return novelty_score, justification

novelty_score, novelty_justification = calculate_novelty_score(grant_keywords, pubmed_articles)

print(f"\n--- Novelty Score ---")
print(f"Novelty Score: {novelty_score:.2f}")
print(f"Justification: {novelty_justification}")


--- Novelty Score ---
Novelty Score: 0.65
Justification: The novelty score is calculated as the proportion of unique keywords from the grant summary that were not found in the titles or abstracts of the top 5 PubMed articles. This moderate score indicates that the grant introduces some novel elements while also building upon existing research. It strikes a balance between innovation and foundational knowledge.
Keywords unique to the grant: local, stakeholders, detection, lab, significance, epidemiology, events, usg, notification, owned, use, americans, learned, governmental, ministries, prosperous, strains, outbreaks, significant, contain, programs, recent, locally, threaten, stronger, home, institutions, partnerships, notice, create, ghs, improving, globally, government, field, highly, hemorrhagic, detect, opportunity, vhf, fevers, efforts, keep, safer, durable, nofo, independently, respond, quickly, combat, applies, abroad, lead, leaders, pathogenic, influenza, updated, prioritizes,